In [0]:
 %run "/Workspace/Users/jeancarlosramoshuamanlazo@hotmail.com/databricksjc/Proceso/0_CONFIGURACION"

In [0]:
from pyspark.sql import functions as F, types as T
from pyspark.sql.window import Window

spark.sql(f"USE CATALOG {catalog_name}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {bronze_catalog}")

In [0]:
RAW_SCHEMAS = {
    "automobile": T.StructType([
        T.StructField("symboling", T.IntegerType(), True),
        T.StructField("normalized_losses", T.StringType(), True),
        T.StructField("make", T.StringType(), True),
        T.StructField("fuel_type", T.StringType(), True),
        T.StructField("aspiration", T.StringType(), True),
        T.StructField("num_of_doors", T.StringType(), True),
        T.StructField("body_style", T.StringType(), True),
        T.StructField("drive_wheels", T.StringType(), True),
        T.StructField("engine_location", T.StringType(), True),
        T.StructField("wheel_base", T.DoubleType(), True),
        T.StructField("length", T.DoubleType(), True),
        T.StructField("width", T.DoubleType(), True),
        T.StructField("height", T.DoubleType(), True),
        T.StructField("curb_weight", T.IntegerType(), True),
        T.StructField("engine_type", T.StringType(), True),
        T.StructField("num_of_cylinders", T.StringType(), True),
        T.StructField("engine_size", T.IntegerType(), True),
        T.StructField("fuel_system", T.StringType(), True),
        T.StructField("bore", T.StringType(), True),
        T.StructField("stroke", T.StringType(), True),
        T.StructField("compression_ratio", T.DoubleType(), True),
        T.StructField("horsepower", T.StringType(), True),
        T.StructField("peak_rpm", T.StringType(), True),
        T.StructField("city_mpg", T.IntegerType(), True),
        T.StructField("highway_mpg", T.IntegerType(), True),
        T.StructField("price", T.StringType(), True),
    ])
}


In [0]:
def read_raw_csv(table_name, path):
    df = (
        spark.read
        .format("csv")
        .option("header", True)
        .option("nullValue", "?")
        .schema(RAW_SCHEMAS[table_name])
        .load(path)
    )

    df = (
        df
        .withColumn("price", F.col("price").cast("double"))
        .withColumn("horsepower", F.col("horsepower").cast("double"))
        .withColumn("peak_rpm", F.col("peak_rpm").cast("double"))
        .withColumn("bore", F.col("bore").cast("double"))
        .withColumn("stroke", F.col("stroke").cast("double"))
        .withColumn("normalized_losses", F.col("normalized_losses").cast("double"))
    )

    return (
        df
        .withColumn("_source_file", F.col("_metadata.file_path"))  
        .withColumn("_ingestion_ts", F.current_timestamp())
        .withColumn("_load_date", F.current_date())
    )

In [0]:
def add_record_hash(df):
    cols = [c for c in df.columns if not c.startswith("_")]
    return df.withColumn(
        "_record_hash",
        F.sha2(F.concat_ws("||", *[F.col(c).cast("string") for c in cols]), 256)
    )


In [0]:
def deduplicate(df):
    w = Window.partitionBy("_record_hash").orderBy(F.col("_ingestion_ts").desc())
    return df.withColumn("rn", F.row_number().over(w)).filter("rn=1").drop("rn")

PROCESO RAW → BRONZE

In [0]:
for table_name, path in source_files.items():
    print(f"Procesando {table_name}")

    df = read_raw_csv(table_name, path)
    df = add_record_hash(df)
    df = deduplicate(df)

    save_delta_table(df, bronze_tables[table_name])

print("RAW → BRONZE OK")